# Emotional scoring
## *Party Above All: The Determinants of Emotional Tone in French Campaign Manifestos* — Machine Learning for NLP
### Author: Salma El Aazdoudi

## Approach

I use three independent methods to score emotional intensity: pyFeel (lexicon-based, French), thomasrenault/emotion (fine-tuned DistilBERT, applied after translation), and mDeBERTa (zero-shot, multilingual). Each method is described in its own section below. I validate these approaches by comparing their outputs on the same texts (convergent validity), conducting manual checks on a sample of manifestos (face validity), and examining whether the results remain consistent at the corpus level.

## 0. Setup

In [1]:
import numpy as np
import pandas as pd
import re
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

# ── numpy pickle patch (required by pyFeel) ───────────────────────────────
if not getattr(np.load, '_pickle_patched', False):
    _original_np_load = np.load
    def _patched_np_load(*args, **kwargs):
        kwargs.setdefault('allow_pickle', True)
        return _original_np_load(*args, **kwargs)
    _patched_np_load._pickle_patched = True
    np.load = _patched_np_load

# ── French tokeniser patch (required by pyFeel) ───────────────────────────
import nltk.tokenize as _nltk_tok
import nltk as _nltk
def _fr_tokenize(text, language='french'):
    return re.findall(r"[a-zàâçéèêëîïôûùüÿæœ'-]+", text.lower())
_nltk_tok.word_tokenize = _fr_tokenize
_nltk.word_tokenize     = _fr_tokenize

In [2]:
# ── Load df_full ─────────────────────────────────────────────────────────
df_full = pd.read_csv("df_full.csv")

In [8]:
df_full.columns

Index(['filename', 'year', 'text', 'n_words', 'id', 'date', 'subject', 'title',
       'contexte-election', 'contexte-tour', 'cote', 'departement',
       'departement-nom', 'departement-insee',
       'identifiant de circonscription', 'images', 'pdf', 'ocr_url',
       'titulaire-nom', 'titulaire-prenom', 'titulaire-sexe', 'titulaire-age',
       'titulaire-age-calcule', 'titulaire-age-tranche',
       'titulaire-profession', 'titulaire-mandat-en-cours',
       'titulaire-mandat-passe', 'titulaire-associations',
       'titulaire-autres-statuts', 'titulaire-soutien', 'titulaire-liste',
       'titulaire-decorations', 'suppleant-nom', 'suppleant-prenom',
       'suppleant-sexe', 'suppleant-age', 'suppleant-age-calcule',
       'suppleant-age-tranche', 'suppleant-profession',
       'suppleant-mandat-en-cours', 'suppleant-mandat-passe',
       'suppleant-associations', 'suppleant-autres-statuts',
       'suppleant-soutien', 'suppleant-liste', 'suppleant-decorations',
       'nuance_matc

In [6]:
df_full["n_words"] = df_full["text"].str.split().str.len()

In [7]:
df_full.groupby("year")["n_words"].agg(["count", "mean", "median", "std", "min", "max"])

,count,mean,median,std,min,max
year,,,,,,
1981,2388,681.177554,608.0,332.941527,128,2765
1993,4058,721.954165,674.0,321.074887,47,2921


## 1. pyFeel (lexicon-based)

pyFeel (Abdaoui et al., 2017) is a French sentiment lexicon. It scores each text on seven dimensions : positivity and Paul Ekman's classification of emotions: `joy`, `fear`, `sadness`, `anger`, `surprise`, `disgust`. Each score is the proportion of emotion-tagged words in that category (bag-of-word approach)

The `positivity` score is a broad aggregate that dominates the output and is excluded from the composite. I add another pyFeel outcome is **`feel_intensity_nrc`** (the mean of the six emotion dimensions).


In [ ]:
!pip install git+https://github.com/AdilZouitine/pyFeel.git -q

from pyFeel import Feel
from tqdm import tqdm
import pandas as pd

EMOTION_COLS = ['positivity', 'joy', 'fear', 'sadness', 'angry', 'surprise', 'disgust']
NRC_COLS     = ['joy', 'fear', 'sadness', 'angry', 'surprise', 'disgust']

def score_pyfeel(text: str) -> dict:
    if not isinstance(text, str) or not text.strip():
        return {e: float('nan') for e in EMOTION_COLS}
    try:
        return Feel(text).emotions()
    except Exception:
        return {e: float('nan') for e in EMOTION_COLS}

# Remplace progress_apply par une boucle tqdm standard
print(f"Scoring {len(df_full):,} texts with pyFeel...")
emotion_records = [
    score_pyfeel(text)
    for text in tqdm(df_full["text"].fillna("").tolist(), desc="Scoring with pyFeel")
]

df_feel = pd.DataFrame(emotion_records, index=df_full.index)
df_feel.columns = [f"feel_{col}" for col in df_feel.columns]
df_full = pd.concat([df_full, df_feel], axis=1)

feel_nrc_cols = [f"feel_{e}" for e in NRC_COLS]
df_full["feel_intensity_nrc"] = df_full[feel_nrc_cols].mean(axis=1)

print(f"\nScored {df_full[feel_nrc_cols].notna().all(axis=1).sum():,} / {len(df_full):,} manifestos")
print(f"\n=== pyFeel statistics ===")
print(df_full[feel_nrc_cols + ["feel_intensity_nrc"]].describe().round(4))
df_full.to_csv("df_full_with_pyfeel.csv", index=False)
print("Saved to df_full_with_pyfeel.csv")

## 2 — Transformer-based Emotion Model (thomasrenault/emotion)

As a second measure, we use **thomasrenault/emotion** (Renault, 2025), a DistilBERT model fine-tuned on ~200k US political texts (tweets, campaign speeches, and congressional floor speeches) annotated with GPT-4o-mini. It predicts 8 independent emotion intensities (sigmoid, range 0–1):

| Label | | Label | |
|-------|--|-------|--|
| `anger` | | `pride` | |
| `sadness` | | `joy` | |
| `fear` | | `gratitude` | |
| `disgust` | | `hope` | |

Unlike pyFeel, this model reads the full sentence and is sensitive to context and negation. It was designed specifically for research on emotion in political communication, making it the closest available model to our domain.

The pipeline is:
```
French text → chunked translation (facebook/nllb-200-1.3B) → emotion model → 8 scores```

### 2-1 — Translation

Texts are translated from French to English using [facebook/nllb-200-1.3B](https://huggingface.co/facebook/nllb-200-1.3B) (Meta AI). Because the professions de foi are long (mean: 1,081 tokens, max: 6,134 tokens), each text is split into overlapping 400-token chunks before translation and the results are concatenated. MarianMT models were initially tested but discarded due to hallucinations and truncation on long texts. The following cell takes 60 hours to run. 

In [ ]:
import torch
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ── 1. Chargement ─────────────────────────────────────────────────────────
MODEL_ID     = "facebook/nllb-200-1.3B"
SRC_LANG     = "fra_Latn"
TGT_LANG     = "eng_Latn"
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
ENG_TOKEN_ID = tokenizer_nllb.convert_tokens_to_ids(TGT_LANG)

# ── 2. Chunking + traduction ──────────────────────────────────────────────
CHUNK_SIZE = 400
OVERLAP    = 20

def chunk_text(text):
    tokenizer_nllb.src_lang = SRC_LANG
    tokens = tokenizer_nllb.encode(text, add_special_tokens=False)
    chunks, start = [], 0
    while start < len(tokens):
        end = min(start + CHUNK_SIZE, len(tokens))
        chunks.append(tokens[start:end])
        if end == len(tokens):
            break
        start += CHUNK_SIZE - OVERLAP
    return [tokenizer_nllb.decode(c) for c in chunks]

def translate_text(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    chunks = chunk_text(text)
    translated = []
    for chunk in chunks:
        try:
            tokenizer_nllb.src_lang = SRC_LANG
            inp = tokenizer_nllb(
                [chunk], return_tensors="pt",
                truncation=True, max_length=512,
            ).to(DEVICE)
            with torch.no_grad():
                out = model_nllb.generate(
                    **inp,
                    forced_bos_token_id  = ENG_TOKEN_ID,
                    max_new_tokens       = 512,
                    num_beams            = 4,
                    no_repeat_ngram_size = 4,
                    repetition_penalty   = 1.2,
                    early_stopping       = True,
                )
            translated.append(tokenizer_nllb.decode(out[0], skip_special_tokens=True))
        except Exception as e:
            print(f"  Chunk failed: {e}")
            translated.append("")
    return " ".join(translated)

# ── 3. Pipeline avec checkpoint ───────────────────────────────────────────
CHECKPOINT = Path("translations_nllb_checkpoint.csv")
SAVE_EVERY  = 50

if CHECKPOINT.exists():
    done = pd.read_csv(CHECKPOINT, index_col=0)["text_en"].dropna()
    done = {k: v for k, v in done.to_dict().items() if str(v).strip() != ""}
    print(f"Resuming — {len(done):,} already done.")
else:
    done = {}

texts = df_full["text"].fillna("").tolist()
ids   = df_full.index.tolist()

remaining_ids   = [i for i in ids if i not in done]
remaining_texts = [texts[ids.index(i)] for i in remaining_ids]
print(f"To translate: {len(remaining_texts):,} texts")

for j, (idx, text) in enumerate(tqdm(
    zip(remaining_ids, remaining_texts),
    total=len(remaining_texts), desc="Translating NLLB"
)):
    done[idx] = translate_text(text)
    if (j + 1) % SAVE_EVERY == 0:
        pd.DataFrame.from_dict(done, orient="index", columns=["text_en"])\
          .to_csv(CHECKPOINT)
        print(f"  Checkpoint saved — {len(done):,} done")

# ── 4. Sauvegarde finale ──────────────────────────────────────────────────
pd.DataFrame.from_dict(done, orient="index", columns=["text_en"])\
  .to_csv(CHECKPOINT)

df_full["text_en"] = pd.Series(done)
df_full.to_csv("df_full_translated_nllb.csv", index=False)

print(f"\nDone — {len(done):,} texts translated.")
print(f"Example FR : {texts[0][:200]}")
print(f"Example EN : {df_full['text_en'].iloc[0][:200]}")

### 2-2 Emotional Scoring with Bert

In [2]:
# ── Load df_translated_nllb ─────────────────────────────────────────────────────────
import pandas as pd
df_full=pd.read_csv("df_full_translated_nllb.csv")

In [2]:
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ── 1. Chargement du modèle ───────────────────────────────────────────────
EMOTION_MODEL_ID = "thomasrenault/emotion"
EMOTIONS         = ["anger", "sadness", "fear", "disgust", "pride", "joy", "gratitude", "hope"]
NRC_SHARED       = ["anger", "sadness", "fear", "disgust", "joy"]

print("Loading emotion model...")
tokenizer_emo = AutoTokenizer.from_pretrained(EMOTION_MODEL_ID)
model_emo     = AutoModelForSequenceClassification.from_pretrained(EMOTION_MODEL_ID)
model_emo.eval()
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
model_emo = model_emo.to(DEVICE)
print(f"Model loaded on {DEVICE}.")

# ── 2. Scoring par chunk ──────────────────────────────────────────────────
def score_batch_chunked(texts: list) -> list:
    results = []
    for text in tqdm(texts, desc="Emotion scoring (chunked)"):
        if not isinstance(text, str) or not text.strip():
            results.append({e: float("nan") for e in EMOTIONS})
            continue
        try:
            # Tokenize full text without truncation
            tokens = tokenizer_emo(
                text,
                truncation=False,
                add_special_tokens=False
            )["input_ids"]

            # Split into non-overlapping chunks of 510
            CHUNK_SIZE = 510
            chunks, start = [], 0
            while start < len(tokens):
                end = min(start + CHUNK_SIZE, len(tokens))
                chunks.append(tokens[start:end])
                start = end

            # Decode each chunk back to text, re-tokenize with special tokens, score
            chunk_scores = []
            for chunk in chunks:
                chunk_text = tokenizer_emo.decode(chunk, skip_special_tokens=True)
                enc = tokenizer_emo(
                    chunk_text,
                    return_tensors="pt",
                    truncation=True,
                    max_length=512,
                    padding=False,
                ).to(DEVICE)
                with torch.no_grad():
                    p = torch.sigmoid(model_emo(**enc).logits).squeeze().tolist()
                chunk_scores.append(dict(zip(EMOTIONS, p)))

            avg = {e: sum(s[e] for s in chunk_scores) / len(chunk_scores) for e in EMOTIONS}
            results.append(avg)

        except Exception as e:
            print(f"Error: {e}")
            results.append({e: float("nan") for e in EMOTIONS})

    return results

# ── 3. Application sur les textes traduits ────────────────────────────────
texts_en = df_full["text_en"].fillna("").tolist()
print(f"Scoring {len(texts_en):,} texts...")
all_scores = score_batch_chunked(texts_en)

# ── 4. Intégration dans df_full ───────────────────────────────────────────
df_tr = pd.DataFrame(all_scores, index=df_full.index)
df_tr.columns = [f"tr_{col}" for col in df_tr.columns]
df_full = pd.concat([df_full, df_tr], axis=1)

tr_nrc_cols = [f"tr_{e}" for e in NRC_SHARED]
df_full["tr_intensity_nrc"] = df_full[tr_nrc_cols].mean(axis=1)

# ── 5. Sauvegarde ─────────────────────────────────────────────────────────
df_full.to_csv("df_full_with_emotions.csv", index=False)

print("\nDone.")
print(f"\n=== thomasrenault/emotion statistics ===")
print(df_full[tr_nrc_cols + ["tr_intensity_nrc"]].describe().round(4))

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading emotion model...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 4225.06it/s]


/opt/python/lib/python3.13/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


...

Emotion scoring (chunked): 100%|█████████▉| 6444/6446 [24:05<00:00,  3.33it/s]

Emotion scoring (chunked): 100%|█████████▉| 6445/6446 [24:05<00:00,  4.07it/s]

Emotion scoring (chunked): 100%|██████████| 6446/6446 [24:05<00:00,  3.49it/s]

Emotion scoring (chunked): 100%|██████████| 6446/6446 [24:05<00:00,  4.46it/s]


Done.

=== thomasrenault/emotion statistics ===
        tr_anger  tr_sadness    tr_fear  tr_disgust     tr_joy  \
count  6446.0000   6446.0000  6446.0000   6446.0000  6446.0000   
mean      0.2682      0.2022     0.2499      0.1848     0.3958   
std       0.2194      0.1453     0.1615      0.1726     0.1317   
min       0.0001      0.0001     0.0001      0.0000     0.0009   
25%       0.0530      0.0679     0.1054      0.0198     0.2932   
50%       0.2442      0.1988     0.2601      0.1429     0.3870   
75%       0.4463      0.3097     0.3823      0.3131     0.4962   
max       0.7499      0.6557     0.6479      0.6387     0.7492   

       tr_intensity_nrc  
count         6446.0000  
mean             0.2602  
std              0.1141  
min              0.0329  
25%              0.1515  
50%              0.2483  
75%              0.3512  
max              0.5336  


## 3 - Zero-Shot Emotion Scoring (mDeBERTa)

I use [MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7](https://huggingface.co/MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7), a multilingual model applied directly to the original French text. For each emotion, the model evaluates whether the hypothesis "This text expresses [e]" is entailed by the input — scores are computed independently per label, allowing multiple emotions to score highly simultaneously. The five labels are anger, fear, sadness, disgust, and joy.

In [3]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import pipeline, logging as transformers_logging
transformers_logging.set_verbosity_error()

# ── 1. Chargement du modèle ───────────────────────────────────────────────
MODEL_ID = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
DEVICE   = 0 if torch.cuda.is_available() else -1

print("Loading mDeBERTa zero-shot model...")
classifier = pipeline(
    "zero-shot-classification",
    model  = MODEL_ID,
    device = DEVICE,
)
tokenizer_zs = classifier.tokenizer
print(f"Model loaded on {'cuda' if DEVICE == 0 else 'cpu'}.")

# ── 2. Labels émotionnels ─────────────────────────────────────────────────
EMOTION_LABELS = ["anger", "fear", "sadness", "disgust", "joy"]

# ── 3. Fonction de scoring ────────────────────────────────────────────────
def score_zero_shot(text: str) -> dict:
    if not isinstance(text, str) or not text.strip():
        return {f"zs_{e}": np.nan for e in EMOTION_LABELS} | {"zs_dominant": None}
    try:
        # Tokenize full text without truncation
        tokens = tokenizer_zs(
            text,
            truncation=False,
            add_special_tokens=False
        )["input_ids"]

        # Split into non-overlapping chunks of 510
        CHUNK_SIZE = 510
        chunks, start = [], 0
        while start < len(tokens):
            end = min(start + CHUNK_SIZE, len(tokens))
            chunks.append(tokens[start:end])
            start = end

        # Score each chunk and average
        chunk_scores = {e: [] for e in EMOTION_LABELS}
        for chunk in chunks:
            chunk_text = tokenizer_zs.decode(chunk, skip_special_tokens=True)
            result = classifier(
                chunk_text,
                candidate_labels    = EMOTION_LABELS,
                multi_label         = True,
                hypothesis_template = "This text expresses {}.",
            )
            for label, score in zip(result["labels"], result["scores"]):
                chunk_scores[label].append(score)

        scores   = {e: np.mean(chunk_scores[e]) for e in EMOTION_LABELS}
        dominant = max(scores, key=scores.get)
        return {f"zs_{e}": scores[e] for e in EMOTION_LABELS} | {"zs_dominant": dominant}

    except Exception as e:
        print(f"  Error: {e}")
        return {f"zs_{e}": np.nan for e in EMOTION_LABELS} | {"zs_dominant": None}

# ── 4. Application sur le corpus ──────────────────────────────────────────
texts = df_full["text"].fillna("").tolist()
print(f"Scoring {len(texts):,} texts...")
all_scores = [score_zero_shot(text) for text in tqdm(texts, desc="Zero-shot scoring")]

# ── 5. Intégration dans df_full ───────────────────────────────────────────
df_zs = pd.DataFrame(all_scores, index=df_full.index)
df_full = pd.concat([df_full, df_zs], axis=1)

zs_nrc_cols = [f"zs_{e}" for e in EMOTION_LABELS]
df_full["zs_intensity_nrc"] = df_full[zs_nrc_cols].mean(axis=1)

# ── 6. Sauvegarde ─────────────────────────────────────────────────────────
df_full.to_csv("df_full_with_zeroshot.csv", index=False)

print("\nDone.")
print(f"\n=== mDeBERTa zero-shot statistics ===")
print(df_full[zs_nrc_cols + ["zs_intensity_nrc"]].describe().round(4))
print(f"\nDominant emotion distribution:")
print(df_full["zs_dominant"].value_counts())

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading mDeBERTa zero-shot model...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 3802.05it/s]

Model loaded on cpu.
Scoring 6,446 texts...


Zero-shot scoring:   0%|          | 0/6446 [00:00<?, ?it/s]

Zero-shot scoring:   0%|          | 1/6446 [02:36<280:09:52, 156.49s/it]

Zero-shot scoring:   0%|          | 1/6446 [03:55<420:51:34, 235.08s/it]

KeyboardInterrupt: 

In [1]:
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu129 --break-system-packages

Looking in indexes: https://download.pytorch.org/whl/nightly/cu129


INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.


INFO: pip is still looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.


INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/657.9 MB ? eta -:--:--

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━ 19/20 [torchvision]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━ 19/20 [torchvision]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━ 19/20 [torchvision]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━ 19/20 [torchvision]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20/20 [torchvision]


In [1]:
import torch
print(torch.cuda.is_available())

ImportError: /opt/python/lib/python3.13/site-packages/torch/lib/libtorch_cuda.so: undefined symbol: ncclCommResume

In [2]:
import pandas as pd

df_feel = pd.read_csv("df_full_with_pyfeel.csv")
df_tr   = pd.read_csv("df_full_with_emotions.csv")
df_zs   = pd.read_csv("df_full_with_zeroshot.csv")

print("=== df_feel columns ===")
print(df_feel.columns.tolist())
print("\n=== df_tr columns ===")
print(df_tr.columns.tolist())
print("\n=== df_zs columns ===")
print(df_zs.columns.tolist())

# Identify columns unique to each scoring file (excluding base columns already in df_feel)
base_cols  = df_feel.columns.tolist()
tr_new     = [c for c in df_tr.columns if c not in base_cols]
zs_new     = [c for c in df_zs.columns if c not in base_cols and c not in tr_new]

df_full_merged = pd.concat([df_feel, df_tr[tr_new], df_zs[zs_new]], axis=1)

print(f"\n=== df_full_merged : {df_full_merged.shape} ===")
print(df_full_merged.columns.tolist())

df_full_merged.to_csv("df_full_merged.csv", index=False)
print("\nSaved to df_full_merged.csv")

=== df_feel columns ===
['filename', 'year', 'text', 'n_words', 'id', 'date', 'subject', 'title', 'contexte-election', 'contexte-tour', 'cote', 'departement', 'departement-nom', 'departement-insee', 'identifiant de circonscription', 'images', 'pdf', 'ocr_url', 'titulaire-nom', 'titulaire-prenom', 'titulaire-sexe', 'titulaire-age', 'titulaire-age-calcule', 'titulaire-age-tranche', 'titulaire-profession', 'titulaire-mandat-en-cours', 'titulaire-mandat-passe', 'titulaire-associations', 'titulaire-autres-statuts', 'titulaire-soutien', 'titulaire-liste', 'titulaire-decorations', 'suppleant-nom', 'suppleant-prenom', 'suppleant-sexe', 'suppleant-age', 'suppleant-age-calcule', 'suppleant-age-tranche', 'suppleant-profession', 'suppleant-mandat-en-cours', 'suppleant-mandat-passe', 'suppleant-associations', 'suppleant-autres-statuts', 'suppleant-soutien', 'suppleant-liste', 'suppleant-decorations', 'nuance_matched', 'position_label', 'ideology_pos', 'dist_center', 'bloc', 'dist_center2', 'year_du


Saved to df_full_merged.csv


## 4- Comparison across the three models 

### 4-1 Do the models agree on the dominant emotion ?

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("df_full_merged.csv")

# ── Colonnes par modèle ───────────────────────────────────────────────────
TR_EMOTIONS   = {"tr_anger": "anger", "tr_sadness": "sadness", "tr_fear": "fear",
                 "tr_disgust": "disgust", "tr_joy": "joy"}
ZS_EMOTIONS   = {"zs_anger": "anger", "zs_fear": "fear", "zs_sadness": "sadness",
                 "zs_disgust": "disgust", "zs_joy": "joy"}
FEEL_EMOTIONS = {"feel_angry": "anger", "feel_fear": "fear", "feel_sadness": "sadness",
                 "feel_disgust": "disgust", "feel_joy": "joy"}

def get_dominant(row, col_map):
    scores = {label: row[col] for col, label in col_map.items()
              if col in row.index and pd.notna(row[col])}
    if not scores:
        return None
    return max(scores, key=scores.get)

# ── Calculer l'émotion dominante pour chaque modèle ──────────────────────
df["dom_tr"]   = df.apply(lambda r: get_dominant(r, TR_EMOTIONS),   axis=1)
df["dom_zs"]   = df.apply(lambda r: get_dominant(r, ZS_EMOTIONS),   axis=1)
df["dom_feel"] = df.apply(lambda r: get_dominant(r, FEEL_EMOTIONS), axis=1)

# Garder les lignes où les 3 modèles ont un résultat
df_valid = df[["dom_tr", "dom_zs", "dom_feel", "bloc"]].dropna()
n = len(df_valid)
print(f"Textes analysés : {n:,}\n")

# ── Taux d'accord 3/3 ────────────────────────────────────────────────────
agree_all = (
    (df_valid["dom_tr"] == df_valid["dom_zs"]) &
    (df_valid["dom_zs"] == df_valid["dom_feel"])
)
print(f"Accord 3/3 (tous d'accord)   : {agree_all.sum():,} / {n:,} ({agree_all.mean()*100:.1f}%)")

# ── Taux d'accord 2/3 ────────────────────────────────────────────────────
agree_tr_zs   = df_valid["dom_tr"]   == df_valid["dom_zs"]
agree_tr_feel = df_valid["dom_tr"]   == df_valid["dom_feel"]
agree_zs_feel = df_valid["dom_zs"]   == df_valid["dom_feel"]
agree_any2 = agree_tr_zs | agree_tr_feel | agree_zs_feel

print(f"Accord 2/3 (au moins 2/3)    : {agree_any2.sum():,} / {n:,} ({agree_any2.mean()*100:.1f}%)")
print(f"  thomasrenault + mDeBERTa   : {agree_tr_zs.sum():,} ({agree_tr_zs.mean()*100:.1f}%)")
print(f"  thomasrenault + pyFeel     : {agree_tr_feel.sum():,} ({agree_tr_feel.mean()*100:.1f}%)")
print(f"  mDeBERTa + pyFeel          : {agree_zs_feel.sum():,} ({agree_zs_feel.mean()*100:.1f}%)")

# ── Désaccord total ───────────────────────────────────────────────────────
disagree_all = ~agree_any2
print(f"\nDésaccord total (0/3)        : {disagree_all.sum():,} / {n:,} ({disagree_all.mean()*100:.1f}%)")

# ── Distribution des émotions dominantes par modèle ──────────────────────
print(f"\n=== Distribution émotions dominantes ===")
for col, name in [("dom_tr", "thomasrenault"), ("dom_zs", "mDeBERTa"), ("dom_feel", "pyFeel")]:
    print(f"\n{name}:")
    print(df_valid[col].value_counts().to_string())

# ── Accord par bloc ───────────────────────────────────────────────────────
print(f"\n=== Accord 3/3 par bloc ===")
df_valid["agree_all"] = agree_all
print(df_valid.groupby("bloc")["agree_all"].agg(["sum", "mean"]).round(3).to_string())

Textes analysés : 6,446

Accord 3/3 (tous d'accord)   : 246 / 6,446 (3.8%)
Accord 2/3 (au moins 2/3)    : 3,192 / 6,446 (49.5%)
  thomasrenault + mDeBERTa   : 807 (12.5%)
  thomasrenault + pyFeel     : 1,341 (20.8%)
  mDeBERTa + pyFeel          : 1,536 (23.8%)

Désaccord total (0/3)        : 3,254 / 6,446 (50.5%)

=== Distribution émotions dominantes ===

thomasrenault:
dom_tr
joy        3699
anger      2051
fear        468
sadness     227
disgust       1

mDeBERTa:
dom_zs
sadness    4169
anger      2053
joy         116
disgust      85
fear         23

pyFeel:
dom_feel
anger      3747
fear       2014
sadness     597
joy          52
disgust      36

=== Accord 3/3 par bloc ===
           sum   mean
bloc                 
ecologist    4  0.007
far_left     9  0.011
far_right   53  0.067
left       132  0.057
right       48  0.025


#### Cross-Model Agreement on Dominant Emotion

Agreement across the three methods is low: all three assign the same dominant emotion 
in only 2.7% of texts (175/6,446), and at least two agree in 53.5% of cases. 

This is not surprising given how differently the three models work, but it has 
implications for interpretation.

###### Pairwise agreement

thomasrenault and pyFeel agree most often (22.5%), followed by thomasrenault and 
mDeBERTa (19.7%), with mDeBERTa and pyFeel agreeing least (16.7%). The fact that 
the two most methodologically different models (a fine-tuned BERT and a word-counting 
lexicon) agree more often than either does with mDeBERTa suggests that mDeBERTa is 
the most discrepant of the three.

###### What the distributions reveal

The dominant emotion distributions tell a clear story about each model's bias:

- **thomasrenault** assigns joy to 53% of texts : the most common label by far. 
This likely reflects the generally aspirational tone of electoral manifestos, which 
the model has learned to associate with positive affect.
- **mDeBERTa** assigns sadness to 57% of texts.
- **pyFeel** assigns anger to 58% of texts. Since most manifestos contain some 
critical or adversarial language, the bag-of-words model defaults to anger whenever 
negative words outnumber positive ones.

Each model has a strong default label: joy for thomasrenault, sadness for mDeBERTa, 
anger for pyFeel.
    
###### What this means for the analysis (and the research questions that build on this scoring)

Low agreement on dominant emotion does not mean the models are useless, it means 
they are measuring different things. The more robust comparison is not which emotion 
dominates, but whether the models agree on *intensity*: do the same candidates score 
high across all three methods? 

### 4-2 Manual Inspection 

In [9]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import re

df = pd.read_csv("df_full_merged.csv")

def clean_text(text):
    if not isinstance(text, str):
        return text
    text = re.sub(r'Sciences?\s*Po\s*/?\s*f(?:onds?|unds?)\s*CEVI\w*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'Sciences?\s*Po\s*/?\s*CEVI\w*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

TR_EMOTIONS   = {"tr_anger": "anger", "tr_sadness": "sadness", "tr_fear": "fear",
                 "tr_disgust": "disgust", "tr_joy": "joy"}
ZS_EMOTIONS   = {"zs_anger": "anger", "zs_fear": "fear", "zs_sadness": "sadness",
                 "zs_disgust": "disgust", "zs_joy": "joy"}
FEEL_EMOTIONS = {"feel_angry": "anger", "feel_fear": "fear", "feel_sadness": "sadness",
                 "feel_disgust": "disgust", "feel_joy": "joy"}

EMOTION_EMOJI = {"anger": "😠", "fear": "😨", "sadness": "😢",
                 "disgust": "🤢", "joy": "😊", "surprise": "😲",
                 "positivity": "✨"}

bloc_colors = {
    "far_left": "#d73027", "left": "#fc8d59",
    "ecologist": "#4dac26", "right": "#91bfdb", "far_right": "#2166ac"
}

def get_dominant(row, col_map):
    scores = {label: row[col] for col, label in col_map.items() if col in row.index and pd.notna(row[col])}
    if not scores:
        return "—", 0
    dominant = max(scores, key=scores.get)
    return dominant, scores[dominant]

# ── Calcul du désaccord ───────────────────────────────────────────────────
df["tr_dom"]   = df.apply(lambda r: get_dominant(r, TR_EMOTIONS)[0],   axis=1)
df["zs_dom"]   = df.apply(lambda r: get_dominant(r, ZS_EMOTIONS)[0],   axis=1)
df["feel_dom"] = df.apply(lambda r: get_dominant(r, FEEL_EMOTIONS)[0], axis=1)

df["full_disagreement"] = (
    (df["tr_dom"] != df["zs_dom"]) &
    (df["zs_dom"] != df["feel_dom"]) &
    (df["tr_dom"] != df["feel_dom"])
)

# ── Diagnostic ───────────────────────────────────────────────────────────
df_dis = df[df["full_disagreement"]].copy()
print(f"Textes en désaccord total : {len(df_dis)}")
print(f"Colonnes TR manquantes : {[c for c in TR_EMOTIONS.keys() if c not in df_dis.columns]}")
print(f"NaN dans TR : {df_dis[list(TR_EMOTIONS.keys())].isna().any(axis=1).sum()}")

# ── Sauvegarde ───────────────────────────────────────────────────────────
def save_comparison_table(data, path="comparison_table.html", n=50, random_state=42):
    # Pas de dropna — on garde tout
    sample = data.sample(min(n, len(data)), random_state=random_state)
    print(f"Nombre de textes dans la table : {len(sample)}")

    rows_html = ""
    for _, row in sample.iterrows():
        tr_dom,   tr_val   = get_dominant(row, TR_EMOTIONS)
        zs_dom,   zs_val   = get_dominant(row, ZS_EMOTIONS)
        feel_dom, feel_val = get_dominant(row, FEEL_EMOTIONS)

        all_three = [tr_dom, zs_dom, feel_dom]
        if len(set(all_three)) == 1:
            agreement = "<span class='agree'>✓ All agree</span>"
        elif len(set(all_three)) == 2:
            agreement = "<span>~ Partial</span>"
        else:
            agreement = "<span class='disagree'>✗ Disagree</span>"

        bloc       = row.get("bloc", "")
        bloc_color = bloc_colors.get(str(bloc), "#999")
        bloc_badge = f"<span style='background:{bloc_color};color:white;padding:2px 6px;border-radius:8px;font-size:10px;'>{bloc}</span>"

        full_text = clean_text(str(row.get("text", ""))).replace("<", "&lt;").replace(">", "&gt;")
        preview   = full_text[:80] + "..."
        text_cell = f"<details><summary>{preview}</summary><div class='full-text'>{full_text}</div></details>"

        def fmt(dom, val):
            emoji = EMOTION_EMOJI.get(dom, "")
            return f"<b>{emoji} {dom}</b><br><small>{val:.2f}</small>"

        rows_html += f"""
        <tr>
            <td>{text_cell}</td>
            <td style='text-align:center'>{bloc_badge}</td>
            <td style='text-align:center'>{fmt(feel_dom, feel_val)}</td>
            <td style='text-align:center'>{fmt(tr_dom, tr_val)}</td>
            <td style='text-align:center'>{fmt(zs_dom, zs_val)}</td>
            <td style='text-align:center'>{agreement}</td>
        </tr>"""

    full_html = f"""<!DOCTYPE html>
<html><head><meta charset='utf-8'>
<title>Emotion Model Comparison</title>
<style>
    body {{ font-family: Arial, sans-serif; padding: 20px; }}
    .cmp-table {{ width:100%; border-collapse:collapse; font-size:12px; }}
    .cmp-table th {{ background:#2c3e50; color:white; padding:8px; text-align:center; }}
    .cmp-table td {{ padding:8px; vertical-align:middle; border-bottom:1px solid #eee; }}
    .cmp-table tr:hover {{ background:#f5f5f5; }}
    .full-text {{ font-size:11px; color:#333; line-height:1.5; white-space:pre-wrap; max-height:150px; overflow-y:auto; width:280px; }}
    details summary {{ cursor:pointer; font-size:11px; color:#2980b9; }}
    .agree {{ background:#d5f5e3; border-radius:4px; padding:2px 4px; }}
    .disagree {{ background:#fde8e8; border-radius:4px; padding:2px 4px; }}
</style>
</head><body>
<h2>Emotion Model Comparison — Full Disagreement Only (n={len(sample)})</h2>
<table class='cmp-table'>
    <tr>
        <th style='width:28%'>Text</th>
        <th>Bloc</th>
        <th>pyFeel</th>
        <th>thomasrenault</th>
        <th>mDeBERTa</th>
        <th>Agreement</th>
    </tr>
    {rows_html}
</table>
</body></html>"""

    with open(path, "w", encoding="utf-8") as f:
        f.write(full_html)
    print(f"Saved to {path}")

save_comparison_table(df_dis, path="comparison_table_disagreement.html", n=50, random_state=42)

Textes en désaccord total : 3254
Colonnes TR manquantes : []
NaN dans TR : 0
Nombre de textes dans la table : 50
Saved to comparison_table_disagreement.html


## Do the Models Agree on Positive vs Negative Sentiment?

Even if the three models disagree on *which* specific emotion dominates, they might still agree on the *valence* of a text — whether it leans positive or negative.

This section tests that weaker form of agreement by collapsing each model's emotion scores into a single **sentiment polarity score**, then comparing them pairwise and at the corpus level.

### Valence mapping

| Dimension | pyFeel columns | thomasrenault columns | mDeBERTa columns |
|-----------|---------------|----------------------|------------------|
| **Positive** | `feel_joy`, `feel_positivity` | `tr_joy`, `tr_hope`, `tr_gratitude`, `tr_pride` | `zs_joy` |
| **Negative** | `feel_angry`, `feel_fear`, `feel_sadness`, `feel_disgust` | `tr_anger`, `tr_fear`, `tr_sadness`, `tr_disgust` | `zs_anger`, `zs_fear`, `zs_sadness`, `zs_disgust` |

Sentiment polarity = mean(positive) − mean(negative).  
Texts with polarity > 0 are labelled **positive**, ≤ 0 are **negative**.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

df = pd.read_csv("df_full_merged.csv")

# ── 1. Valence column definitions ─────────────────────────────────────────
FEEL_POS  = ["feel_joy", "feel_positivity"]
FEEL_NEG  = ["feel_angry", "feel_fear", "feel_sadness", "feel_disgust"]

TR_POS    = ["tr_joy", "tr_hope", "tr_gratitude", "tr_pride"]
TR_NEG    = ["tr_anger", "tr_fear", "tr_sadness", "tr_disgust"]

ZS_POS    = ["zs_joy"]
ZS_NEG    = ["zs_anger", "zs_fear", "zs_sadness", "zs_disgust"]

def existing(cols):
    return [c for c in cols if c in df.columns]

# ── 2. Compute polarity scores ─────────────────────────────────────────────
df["feel_polarity"] = df[existing(FEEL_POS)].mean(axis=1) - df[existing(FEEL_NEG)].mean(axis=1)
df["tr_polarity"]   = df[existing(TR_POS)].mean(axis=1)   - df[existing(TR_NEG)].mean(axis=1)
df["zs_polarity"]   = df[existing(ZS_POS)].mean(axis=1)   - df[existing(ZS_NEG)].mean(axis=1)

# ── 3. Binary sentiment labels ─────────────────────────────────────────────
df["feel_sent"] = (df["feel_polarity"] > 0).map({True: "positive", False: "negative"})
df["tr_sent"]   = (df["tr_polarity"]   > 0).map({True: "positive", False: "negative"})
df["zs_sent"]   = (df["zs_polarity"]   > 0).map({True: "positive", False: "negative"})

df_valid = df[["feel_polarity", "tr_polarity", "zs_polarity",
               "feel_sent",    "tr_sent",    "zs_sent",    "bloc"]].dropna()
n = len(df_valid)
print(f"Texts with all three polarity scores: {n:,}\n")

# ── 4. Binary agreement rates ──────────────────────────────────────────────
agree_all = (
    (df_valid["feel_sent"] == df_valid["tr_sent"]) &
    (df_valid["tr_sent"]   == df_valid["zs_sent"])
)
agree_ft = df_valid["feel_sent"] == df_valid["tr_sent"]
agree_fz = df_valid["feel_sent"] == df_valid["zs_sent"]
agree_tz = df_valid["tr_sent"]   == df_valid["zs_sent"]

print("=== Binary sentiment agreement (positive / negative) ===")
print(f"All 3 agree           : {agree_all.sum():,} / {n:,}  ({agree_all.mean()*100:.1f}%)")
print(f"pyFeel + thomasrenault: {agree_ft.sum():,}  ({agree_ft.mean()*100:.1f}%)")
print(f"pyFeel + mDeBERTa     : {agree_fz.sum():,}  ({agree_fz.mean()*100:.1f}%)")
print(f"thomasrenault + mDeBERTa: {agree_tz.sum():,}  ({agree_tz.mean()*100:.1f}%)")

# ── 5. Sentiment distribution per model ───────────────────────────────────
print("\n=== Sentiment distribution ===")
for col, name in [("feel_sent", "pyFeel"), ("tr_sent", "thomasrenault"), ("zs_sent", "mDeBERTa")]:
    counts = df_valid[col].value_counts()
    print(f"  {name}: positive={counts.get('positive',0):,}  negative={counts.get('negative',0):,}")

Texts with all three polarity scores: 6,446

=== Binary sentiment agreement (positive / negative) ===
All 3 agree           : 711 / 6,446  (11.0%)
pyFeel + thomasrenault: 4,996  (77.5%)
pyFeel + mDeBERTa     : 724  (11.2%)
thomasrenault + mDeBERTa: 2,148  (33.3%)

=== Sentiment distribution ===
  pyFeel: positive=6,446  negative=0
  thomasrenault: positive=4,996  negative=1,450
  mDeBERTa: positive=724  negative=5,722


# Validation with Bertopic : Can mDeBERTa distinguish topic from tone? , example with unemployment

In [5]:
!pip install bertopic sentence-transformers umap-learn hdbscan spacy
!python -m spacy download fr_core_news_md

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/571.3 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 20.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/10.5 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 90.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 kB 55.1 MB/s  0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/4.2 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 87.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/3.3 MB ? eta -:--:--

...

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/45.8 MB ? eta -:--:--

     ━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/45.8 MB 113.0 MB/s eta 0:00:01

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━ 38.8/45.8 MB 112.8 MB/s eta 0:00:01

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 MB 92.7 MB/s  0:00:00


✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_md')


In [6]:
import pandas as pd
import numpy as np
import re
import spacy
import warnings
warnings.filterwarnings("ignore")

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("df_full_merged.csv")
TEXT_COL = "text"  # <- adapte si besoin

print(df.shape)
print(df.columns.tolist())
print(df[TEXT_COL].iloc[0][:300])

(6446, 78)
['filename', 'year', 'text', 'n_words', 'id', 'date', 'subject', 'title', 'contexte-election', 'contexte-tour', 'cote', 'departement', 'departement-nom', 'departement-insee', 'identifiant de circonscription', 'images', 'pdf', 'ocr_url', 'titulaire-nom', 'titulaire-prenom', 'titulaire-sexe', 'titulaire-age', 'titulaire-age-calcule', 'titulaire-age-tranche', 'titulaire-profession', 'titulaire-mandat-en-cours', 'titulaire-mandat-passe', 'titulaire-associations', 'titulaire-autres-statuts', 'titulaire-soutien', 'titulaire-liste', 'titulaire-decorations', 'suppleant-nom', 'suppleant-prenom', 'suppleant-sexe', 'suppleant-age', 'suppleant-age-calcule', 'suppleant-age-tranche', 'suppleant-profession', 'suppleant-mandat-en-cours', 'suppleant-mandat-passe', 'suppleant-associations', 'suppleant-autres-statuts', 'suppleant-soutien', 'suppleant-liste', 'suppleant-decorations', 'nuance_matched', 'position_label', 'ideology_pos', 'dist_center', 'bloc', 'dist_center2', 'year_dummy', 'feel_p

In [3]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/27.8 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━ 15.5/27.8 MB 108.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 91.2 MB/s  0:00:00


Note: you may need to restart the kernel to use updated packages.


In [7]:
try:
    nlp = spacy.load("fr_core_news_md")
except OSError:
    import subprocess
    subprocess.run(["python", "-m", "spacy", "download", "fr_core_news_md"])
    nlp = spacy.load("fr_core_news_md")

GENERAL_STOPWORDS = set(nlp.Defaults.stop_words)

raw_custom_stopwords = {
    "monsieur","madame","messieurs","mesdames","mademoiselle",
    "candidat","candidate","candidats","gauche","droite",
    "election","vote","voter","circonscription","departement",
    "republique","sciences","po","cevipof","française","fonds",
    "liberte","egalite","fraternite","electeur","electrice",
    "electeurs","electrices","parti","partis",
    "janvier","fevrier","mars","avril","mai","juin","juillet",
    "aout","septembre","octobre","novembre","decembre",
    "politique","france","français","française","national","nationale",
    "election","elections","electoral","electorale","legislative",
    "presidentielle","municipale","majorite","minorite",
    "gouvernement","opposition","parlement","assemblee","senat",
    "depute","deputee","suppleant","suppleante","programme",
    "commun","action","moyen","mesure","projets","projet",
    "avenir","changement","ensemble","pays","citoyen","citoyenne",
}

_nlp_sw = spacy.load("fr_core_news_md", disable=["parser","ner"])
custom_stopwords_lemmatized = set()
for doc in _nlp_sw.pipe(list(raw_custom_stopwords), disable=["parser","ner"]):
    for token in doc:
        custom_stopwords_lemmatized.add(token.lemma_.lower())

custom_stopwords_lemmatized.update({
    "departement","circonscription","republique","liberte","egalite",
    "fraternite","election","legislative","legislatives","president",
    "suppleant","suppleants","suppleante","suppleantes",
})

print(f"Stopwords custom : {len(custom_stopwords_lemmatized)}")

# ← AJOUT ICI
STOPWORDS = list(GENERAL_STOPWORDS | custom_stopwords_lemmatized)
print(f"STOPWORDS total : {len(STOPWORDS)}")

Stopwords custom : 71
STOPWORDS total : 578


In [10]:
ALLOWED_POS   = {"NOUN", "ADJ", "VERB"}   # VERB ajouté vs ton step1
VALID_PATTERN = re.compile(
    r'^[a-zA-ZàâäéèêëïîôöùûüÿçÀÂÄÉÈÊËÏÎÔÖÙÛÜŸÇ\-]+$'
)

def basic_clean(text):
    """Nettoyage minimal avant spaCy."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\b(\w+)\s+\1\b', r'\1', text, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', text).strip()

def extract_lemmas(texts):
    """Lemmatisation + filtre POS + filtre stopwords."""
    results = []
    for doc in nlp.pipe(texts, batch_size=100):
        lemmas = []
        for token in doc:
            if (token.is_alpha
                    and not token.is_space
                    and token.pos_ in ALLOWED_POS
                    and token.ent_type_ != "PER"
                    and token.text.lower() not in GENERAL_STOPWORDS
                    and token.lemma_.lower() not in GENERAL_STOPWORDS
                    and token.lemma_.lower() not in custom_stopwords_lemmatized
                    and len(token.lemma_) > 3
                    and VALID_PATTERN.match(token.lemma_)):
                lemmas.append(token.lemma_.lower())
        results.append(" ".join(lemmas))
    return results

In [11]:
print("Nettoyage basique...")
df["cleaned_text"] = df[TEXT_COL].apply(basic_clean)

print("Lemmatisation (peut prendre ~10 min)...")
texts = df["cleaned_text"].astype(str).tolist()
lemmatized = []
for i in range(0, len(texts), 200):
    batch = extract_lemmas(texts[i:i+200])
    lemmatized.extend(batch)
    if (i // 200) % 5 == 0:
        print(f"  {min(i+200, len(texts))}/{len(texts)} docs...")

df["text_clean"] = lemmatized

# Retire les textes trop courts après nettoyage
df = df[df["text_clean"].str.split().str.len() >= 15].reset_index(drop=True)

empty = (df["text_clean"].str.strip() == "").sum()
print(f"Done. Textes vides : {empty} — Corpus final : {len(df):,} docs")
print(f"Exemple : {df['text_clean'].iloc[0][:200]}")

Nettoyage basique...


Lemmatisation (peut prendre ~10 min)...


  200/6446 docs...


  1200/6446 docs...


  2200/6446 docs...


  3200/6446 docs...


  4200/6446 docs...


  5200/6446 docs...


  6200/6446 docs...


Done. Textes vides : 0 — Corpus final : 6,446 docs
Exemple : législatif cheminot lutte suppléant employer travailleur travailleur lutte ouvrier formation présenter candidature élection présidentiel solliciter suffrage élection présenter côté sigle majorité côté


In [12]:
import pickle

with open("bertopic_preprocessed.pkl", "wb") as f:
    pickle.dump(df, f)

print(f"Sauvegardé : {len(df):,} docs")

Sauvegardé : 6,446 docs


In [8]:
import pickle

with open("bertopic_preprocessed.pkl", "rb") as f:
    df = pickle.load(f)

print(f"Chargé : {len(df):,} docs")
print(df["text_clean"].iloc[0][:200])

Chargé : 6,446 docs
législatif cheminot lutte suppléant employer travailleur travailleur lutte ouvrier formation présenter candidature élection présidentiel solliciter suffrage élection présenter côté sigle majorité côté


In [9]:
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

docs = df["text_clean"].tolist()

# Embeddings
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings = embedding_model.encode(docs, show_progress_bar=True)
print(f"Embeddings shape: {embeddings.shape}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7152.24it/s]

Batches:   0%|          | 0/202 [00:00<?, ?it/s]

Batches:   0%|          | 1/202 [00:08<29:45,  8.88s/it]

Batches:   1%|          | 2/202 [00:16<26:51,  8.06s/it]

...

Batches:  86%|████████▌ | 173/202 [11:13<02:21,  4.89s/it]

Batches:  86%|████████▌ | 174/202 [11:21<02:37,  5.61s/it]

Batches:  87%|████████▋ | 175/202 [11:28<02:47,  6.22s/it]

Batches:  87%|████████▋ | 175/202 [11:31<01:46,  3.95s/it]

KeyboardInterrupt: 

In [14]:
import numpy as np
np.save("bertopic_embeddings.npy", embeddings)
print("Embeddings sauvegardés.")

Embeddings sauvegardés.


In [4]:
import numpy as np

embeddings = np.load("bertopic_embeddings.npy")
docs = df["text_clean"].tolist()
print(f"Embeddings chargés : {embeddings.shape}")

Embeddings chargés : (6446, 384)


In [6]:
from sklearn.feature_extraction.text import CountVectorizer

In [18]:
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

umap_model = UMAP(
    n_neighbors=10,       # réduit : clusters plus locaux
    n_components=10,      # augmenté : plus de dimensions = meilleure séparation
    metric="cosine",
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=20,  # réduit : accepte des clusters plus petits
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="leaf",  # "leaf" donne plus de topics que "eom"
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    stop_words=STOPWORDS,
    min_df=5
)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    calculate_probabilities=False,
    nr_topics="auto",
    verbose=True
)

topics, probs = topic_model.fit_transform(docs, embeddings)
df["topic"] = topics

print(f"{len(set(topics))} topics trouvés (dont -1 = outliers)")
print(topic_model.get_topic_info().to_string())

2026-04-29 10:59:49,288 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


2026-04-29 11:00:02,013 - BERTopic - Dimensionality - Completed ✓


2026-04-29 11:00:02,016 - BERTopic - Cluster - Start clustering the reduced embeddings


2026-04-29 11:00:02,334 - BERTopic - Cluster - Completed ✓


2026-04-29 11:00:02,335 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.


...

2026-04-29 11:00:03,278 - BERTopic - Topic reduction - Reducing number of topics


2026-04-29 11:00:03,294 - BERTopic - Representation - Fine-tuning topics using representation models.


2026-04-29 11:00:04,265 - BERTopic - Representation - Completed ✓


2026-04-29 11:00:04,267 - BERTopic - Topic reduction - Reduced number of topics from 72 to 5


5 topics trouvés (dont -1 = outliers)
   Topic  Count                                    Name                                                                                  Representation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [10]:
topic_info = topic_model.get_topic_info()
topic_info = topic_info[topic_info["Topic"] != -1]

for _, row in topic_info.head(15).iterrows():
    tid   = row["Topic"]
    words = [w for w, _ in topic_model.get_topic(tid)]
    print(f"Topic {tid:3d} ({row['Count']:5.0f} textes) : {', '.join(words[:8])}")

Topic   0 ( 5920 textes) : social, majorité, député, emploi, falloir, communiste, pouvoir, vouloir
Topic   1 (  235 textes) : patron, payer, maintenir, travailleur, entreprise, jeter, sacrifice, faire
Topic   2 (   79 textes) : mettre, refus, maintien, instauration, etat, formation, création, défense
Topic   3 (   63 textes) : travailleur, ouvrier, unité, école, public, guerre, paysan, salaire
Topic   4 (   40 textes) : écologie, ecologiste, écologiste, ecologie, environnement, entente, humain, chance


In [ ]:
TR_EMOTIONS   = {"tr_anger": "anger", "tr_sadness": "sadness", "tr_fear": "fear",
                 "tr_disgust": "disgust", "tr_joy": "joy"}
ZS_EMOTIONS   = {"zs_anger": "anger", "zs_fear": "fear", "zs_sadness": "sadness",
                 "zs_disgust": "disgust", "zs_joy": "joy"}
FEEL_EMOTIONS = {"feel_angry": "anger", "feel_fear": "fear", "feel_sadness": "sadness",
                 "feel_disgust": "disgust", "feel_joy": "joy"}

def dominant_emotion(row, emotion_dict):
    scores = {label: row[col] for col, label in emotion_dict.items()}
    return max(scores, key=scores.get)

df["dom_tr"]   = df.apply(lambda r: dominant_emotion(r, TR_EMOTIONS),   axis=1)
df["dom_zs"]   = df.apply(lambda r: dominant_emotion(r, ZS_EMOTIONS),   axis=1)
df["dom_feel"] = df.apply(lambda r: dominant_emotion(r, FEEL_EMOTIONS), axis=1)

print(df[["dom_tr", "dom_zs", "dom_feel"]].value_counts().head(10))

In [ ]:
def emotion_dist_by_topic(df, dom_col, model_name, top_n=20):
    sub      = df[df["topic"] != -1]
    top_topics = sub["topic"].value_counts().head(top_n).index
    sub      = sub[sub["topic"].isin(top_topics)]
    ct       = pd.crosstab(sub["topic"], sub[dom_col], normalize="index").round(3)
    ct.index.name = f"topic ({model_name})"
    return ct

ct_tr   = emotion_dist_by_topic(df, "dom_tr",   "DistilBERT")
ct_zs   = emotion_dist_by_topic(df, "dom_zs",   "mDeBERTa")
ct_feel = emotion_dist_by_topic(df, "dom_feel", "FEEL")

print(ct_tr)